In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.inspection import permutation_importance
from lightgbm import LGBMClassifier

In [ ]:
def load_preprocessed(path):
    df = pd.read_csv(path)

    y = df['Y_Quality'].copy()
    y_class = df['Y_Class'].copy()
    temp = df[['PRODUCT_ID', 'PRODUCT_CODE', 'TIMESTAMP']].copy()

    drop_cols = ['Y_Quality', 'Y_Class', 'PRODUCT_ID', 'PRODUCT_CODE', 'TIMESTAMP']
    X = df.drop(columns=drop_cols)

    return X, y, y_class, temp

In [ ]:
def compute_cv_importance(X, y_class, n_splits=5, n_repeats=30, scoring='f1_macro', random_state=42):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    fold_importances = []
    positive_fold_count = pd.Series(0, index=X.columns)

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y_class)):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y_class.iloc[train_idx], y_class.iloc[val_idx]

        model = LGBMClassifier(class_weight='balanced', random_state=random_state, verbose=-1)
        model.fit(X_train, y_train)

        result = permutation_importance(
            model, X_val, y_val,
            scoring=scoring,
            n_repeats=n_repeats,
            random_state=random_state,
            n_jobs=-1
        )

        fold_importance = pd.Series(result.importances_mean, index=X.columns)
        fold_importances.append(fold_importance)
        positive_fold_count += (fold_importance > 0).astype(int)

    importance_df = pd.concat(fold_importances, axis=1)
    importance_df.columns = [f'fold_{i}' for i in range(n_splits)]
    importance_df['mean_importance'] = importance_df.mean(axis=1)
    importance_df['positive_folds'] = positive_fold_count

    return importance_df

In [ ]:
def select_features(importance_df, threshold=0, top_n=None):
    selected = importance_df[importance_df['mean_importance'] > threshold]

    if top_n is not None:
        selected = selected.sort_values('mean_importance', ascending=False).head(top_n)

    return selected.index.tolist()

In [ ]:
def select_by_importance(path, output_path, n_splits=5, n_repeats=30, scoring='f1_macro',
                          threshold=0, top_n=None, random_state=42):
    X, y, y_class, temp = load_preprocessed(path)

    importance_df = compute_cv_importance(
        X, y_class,
        n_splits=n_splits, n_repeats=n_repeats, scoring=scoring, random_state=random_state
    )

    selected_cols = select_features(importance_df, threshold=threshold, top_n=top_n)

    print('=' * 50)
    print(f'원본 피처 수: {X.shape[1]}')
    print(f'선택된 피처 수: {len(selected_cols)}')
    print('=' * 50)

    X_selected = X[selected_cols]
    result_df = pd.concat([X_selected, y_class, y, temp], axis=1)
    result_df.to_csv(output_path, index=False, encoding='utf-8-sig')

    return X_selected, y, y_class, temp, importance_df

In [ ]:
X_A_selected, y_A, y_class_A, temp_A, importance_A = select_by_importance(
    'A_31_preprocessed.csv', 'A_31_selected.csv'
)

In [ ]:
X_TO_selected, y_TO, y_class_TO, temp_TO, importance_TO = select_by_importance(
    'TO_31_preprocessed.csv', 'TO_31_selected.csv'
)

In [ ]:
importance_A.sort_values('mean_importance', ascending=False).head(20)

In [ ]:
importance_TO.sort_values('mean_importance', ascending=False).head(20)